# Real Data Predictions
Load training data → train model → preprocess REAL_DATA → predict → export

In [17]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

## Step 1 — Load & Inspect REAL_DATA

In [18]:
df_real = pd.read_csv('../data/REAL_DATA.csv')
print('Shape:', df_real.shape)
print('Columns:', df_real.columns.tolist())
print('\nDtypes:\n', df_real.dtypes)
print('\nMissing values:\n', df_real.isna().sum())
df_real.head()

Shape: (71205, 9)
Columns: ['index', 'store_ID', 'day_of_week', 'date', 'nb_customers_on_day', 'open', 'promotion', 'state_holiday', 'school_holiday']

Dtypes:
 index                  int64
store_ID               int64
day_of_week            int64
date                     str
nb_customers_on_day    int64
open                   int64
promotion              int64
state_holiday            str
school_holiday         int64
dtype: object

Missing values:
 index                  0
store_ID               0
day_of_week            0
date                   0
nb_customers_on_day    0
open                   0
promotion              0
state_holiday          0
school_holiday         0
dtype: int64


,index,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday
0,272371,415,7,01/03/2015,0,0,0,0,0
1,558468,27,7,29/12/2013,0,0,0,0,0
2,76950,404,3,19/03/2014,657,1,1,0,0
3,77556,683,2,29/01/2013,862,1,0,0,0
4,456344,920,3,19/03/2014,591,1,1,0,0


In [20]:
print('state_holiday unique:', df_real['state_holiday'].unique())
print('open unique:         ', df_real['open'].unique())
print('Closed rows:         ', len(df_real[df_real['open'] == 0]))
print('Open rows:           ', len(df_real[df_real['open'] == 1]))

state_holiday unique: <StringArray>
['0', 'a', 'b', 'c']
Length: 4, dtype: str
open unique:          [0 1]
Closed rows:          12100
Open rows:            59105


## Step 2 — Preprocess REAL_DATA

In [21]:
# Drop index column (same as Unnamed: 0 in training)
df_real = df_real.drop(columns=['index'])
print('Dropped index column. Shape:', df_real.shape)

Dropped index column. Shape: (71205, 8)


In [22]:
# Fix date — REAL_DATA uses DD/MM/YYYY format (different from training YYYY-MM-DD)
df_real['date'] = pd.to_datetime(df_real['date'], dayfirst=True)
df_real['year']       = df_real['date'].dt.year
df_real['month']      = df_real['date'].dt.month
df_real['day']        = df_real['date'].dt.day
df_real['week']       = df_real['date'].dt.isocalendar().week.astype(int)
df_real['is_weekend'] = df_real['date'].dt.dayofweek.isin([5, 6]).astype(int)
df_real = df_real.drop(columns=['date'])
print('Date features extracted. Shape:', df_real.shape)

Date features extracted. Shape: (71205, 12)


In [23]:
# Encode state_holiday — One-Hot Encoding (required for Linear Regression)
# Must cast to str BEFORE get_dummies
df_real['state_holiday'] = df_real['state_holiday'].astype(str)
df_real = pd.get_dummies(df_real, columns=['state_holiday'], prefix='holiday', drop_first=True)

# Convert boolean OHE columns to int
for col in ['holiday_a', 'holiday_b', 'holiday_c']:
    if col in df_real.columns:
        df_real[col] = df_real[col].astype(int)

print('Encoding done. Columns:', df_real.columns.tolist())
df_real.head()

Encoding done. Columns: ['store_ID', 'day_of_week', 'nb_customers_on_day', 'open', 'promotion', 'school_holiday', 'year', 'month', 'day', 'week', 'is_weekend', 'holiday_a', 'holiday_b', 'holiday_c']


,store_ID,day_of_week,nb_customers_on_day,open,promotion,school_holiday,year,month,day,week,is_weekend,holiday_a,holiday_b,holiday_c
0,415,7,0,0,0,0,2015,3,1,9,1,0,0,0
1,27,7,0,0,0,0,2013,12,29,52,1,0,0,0
2,404,3,657,1,1,0,2014,3,19,12,0,0,0,0
3,683,2,862,1,0,0,2013,1,29,5,0,0,0,0
4,920,3,591,1,1,0,2014,3,19,12,0,0,0,0


## Step 3 — Load & Preprocess TRAINING DATA (to train the model)

In [24]:
df_train = pd.read_csv('../data/training.csv')
print('Training shape:', df_train.shape)

Training shape: (640840, 10)


In [25]:
# Drop index column
df_train = df_train.drop(columns=['Unnamed: 0'])

# Drop anomaly: open + customers > 0 + sales = 0
anomaly_mask = (df_train['open'] == 1) & (df_train['nb_customers_on_day'] > 0) & (df_train['sales'] == 0)
df_train = df_train[~anomaly_mask]
print(f'Removed {anomaly_mask.sum()} anomalous row(s)')

# Keep only open stores (closed = sales always 0, not useful for learning patterns)
df_train = df_train[df_train['open'] == 1].copy()
print(f'Rows after removing closed stores: {len(df_train)}')

Removed 1 anomalous row(s)
Rows after removing closed stores: 532015


In [26]:
# Date: training uses YYYY-MM-DD format
df_train['date'] = pd.to_datetime(df_train['date'])
df_train['year']       = df_train['date'].dt.year
df_train['month']      = df_train['date'].dt.month
df_train['day']        = df_train['date'].dt.day
df_train['week']       = df_train['date'].dt.isocalendar().week.astype(int)
df_train['is_weekend'] = df_train['date'].dt.dayofweek.isin([5, 6]).astype(int)
df_train = df_train.drop(columns=['date'])

# Encode state_holiday
df_train['state_holiday'] = df_train['state_holiday'].astype(str)
df_train = pd.get_dummies(df_train, columns=['state_holiday'], prefix='holiday', drop_first=True)

# Convert boolean OHE columns to int
for col in ['holiday_a', 'holiday_b', 'holiday_c']:
    if col in df_train.columns:
        df_train[col] = df_train[col].astype(int)

print('Training preprocessing done. Shape:', df_train.shape)
df_train.head()

Training preprocessing done. Shape: (532015, 15)


,store_ID,day_of_week,nb_customers_on_day,open,promotion,school_holiday,sales,year,month,day,week,is_weekend,holiday_a,holiday_b,holiday_c
0,366,4,517,1,0,0,4422,2013,4,18,16,0,0,0,0
1,394,6,694,1,0,0,8297,2015,4,11,15,1,0,0,0
2,807,4,970,1,1,0,9729,2013,8,29,35,0,0,0,0
3,802,2,473,1,1,0,6513,2013,5,28,22,0,0,0,0
4,726,4,1068,1,1,0,10882,2013,10,10,41,0,0,0,0


## Step 4 — Train Linear Regression

In [27]:
# Define features and target
X = df_train.drop(columns=['sales'])
y = df_train['sales']

# Save feature columns to align REAL_DATA later
feature_columns = X.columns.tolist()
print('Features:', feature_columns)

Features: ['store_ID', 'day_of_week', 'nb_customers_on_day', 'open', 'promotion', 'school_holiday', 'year', 'month', 'day', 'week', 'is_weekend', 'holiday_a', 'holiday_b', 'holiday_c']


In [28]:
# Train / Validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)
print(f'Train size: {len(X_train)} | Validation size: {len(X_val)}')

Train size: 399011 | Validation size: 133004


In [29]:
# Train model
lr = LinearRegression()
lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [30]:
# Evaluate
y_train_pred = lr.predict(X_train)
y_val_pred   = lr.predict(X_val)

r2_train      = r2_score(y_train, y_train_pred)
r2_validation = r2_score(y_val,   y_val_pred)
rmse          = np.sqrt(mean_squared_error(y_val, y_val_pred))

print('=== Linear Regression Results ===')
print(f'R2 Train      : {r2_train:.4f}')
print(f'R2 Validation : {r2_validation:.4f}  ← what we expect on real data')
print(f'RMSE          : {rmse:.2f}')

=== Linear Regression Results ===
R2 Train      : 0.7355
R2 Validation : 0.7351  ← what we expect on real data
RMSE          : 1600.72


## Step 5 — Predict on REAL_DATA

In [31]:
# Align REAL_DATA columns to exactly match training features
df_real_aligned = df_real.reindex(columns=feature_columns, fill_value=0)
print('Aligned shape:', df_real_aligned.shape)
print('Columns match:', df_real_aligned.columns.tolist() == feature_columns)

Aligned shape: (71205, 14)
Columns match: True


In [32]:
# Predict
predictions = np.zeros(len(df_real))

# Open stores: use model
open_mask = df_real['open'] == 1
predictions[open_mask] = lr.predict(df_real_aligned[open_mask])

# Closed stores: always 0 (business rule)
predictions[~open_mask] = 0

# Round to integers (sales are whole numbers)
predictions = predictions.clip(min=0).round().astype(int)

print('Predictions done.')
print('Sample predictions:', predictions[:10])

Predictions done.
Sample predictions: [    0     0  7023  6687  6535  4953  5404 10262     0  6654]


## Step 6 — Export to CSV

In [34]:
# Load original REAL_DATA to preserve original format
df_output = pd.read_csv('../data/REAL_DATA.csv')
df_output['sales'] = predictions

# Export
df_output.to_csv('../data/group_4.csv', index=False)

print('✅ Exported group_4.csv')
print('Shape:', df_output.shape)
df_output[['store_ID', 'date', 'open', 'sales']].head(10)

✅ Exported group_4.csv
Shape: (71205, 10)


,store_ID,date,open,sales
0,415,01/03/2015,0,0
1,27,29/12/2013,0,0
2,404,19/03/2014,1,7023
3,683,29/01/2013,1,6687
4,920,19/03/2014,1,6535
5,758,26/06/2014,1,4953
6,563,16/02/2015,1,5404
7,930,22/11/2014,1,10262
8,756,04/06/2015,0,0
9,49,13/01/2015,1,6654


In [35]:
# Sanity check: closed stores should have sales = 0
print('Closed stores with sales > 0:', len(df_output[(df_output['open'] == 0) & (df_output['sales'] > 0)]))
print('Open stores prediction stats:')
print(df_output[df_output['open'] == 1]['sales'].describe())

Closed stores with sales > 0: 0
Open stores prediction stats:
count    59105.000000
mean      6955.803485
std       2683.235391
min        538.000000
25%       5181.000000
50%       6515.000000
75%       8052.000000
max      34607.000000
Name: sales, dtype: float64
